# Модуль 01 · Задача 1.2. Softmax двух классов = сигмоида

Покажите, что softmax двух классов равен сигмоиде от разности логитов.

`modules/01_notation.md`

In [ ]:
import numpy as np

sigma = lambda z: 1 / (1 + np.exp(-z))
softmax = lambda z: np.exp(z) / np.exp(z).sum()

In [ ]:
for z1, z2 in [(3, 1), (10, 8), (-2, 5)]:
    print(z1, z2, softmax(np.array([z1, z2]))[0], sigma(z1 - z2))

## Саммари

**Что разобрали:**
- **Суть задачи:** бинарный классификатор можно сделать двумя способами. Первый: 2 логита и softmax. Второй: 1 логит и сигмоида. Задача доказывает, что они эквивалентны.
- **Вывод:**

$$\frac{e^{z_1}}{e^{z_1}+e^{z_2}} = \frac{1}{1+\dfrac{e^{z_2}}{e^{z_1}}} = \frac{1}{1+e^{\,z_2-z_1}} = \frac{1}{1+e^{-(z_1-z_2)}} = \sigma(z_1 - z_2)$$

  Шаги: поделили числитель и знаменатель на $e^{z_1}$, затем $\frac{e^a}{e^b} = e^{a-b}$, затем вынесли минус и узнали сигмоиду.
- **Числа:** при $z_1 = 3$, $z_2 = 1$ softmax класса 1 и $\sigma(2)$ дают одно и то же, $\approx 0.88$.
- **Feynman:** пары $(3, 1)$ и $(10, 8)$ дают одинаковую вероятность, потому что softmax двух чисел видит только их разность.
- **ML:** второй выход избыточен, поэтому в бинарной классификации ставят 1 выход с сигмоидой. Сдвиг не влияет: $\mathrm{softmax}(\mathbf{z}+c) = \mathrm{softmax}(\mathbf{z})$, и на этом построен стабильный softmax (модуль 13).

### Теория по трудным местам

**Аргумент и показатель — разные вещи.** В $\sigma(t) = \frac{1}{1+e^{-t}}$ показатель экспоненты равен $-t$. Чтобы узнать сигмоиду в $\frac{1}{1+e^{A}}$, решаем уравнение $-t = A$ и получаем $t = -A$. Отсюда при $A = z_2 - z_1$ имеем $t = z_1 - z_2$.

**Проверка смыслом:** сигмоида возрастает. Вероятность класса 1 должна расти с $z_1$, значит, аргумент тоже должен расти с $z_1$. Это $z_1 - z_2$, а не $z_2 - z_1$.

**Симметрия сигмоиды:** $\sigma(-z) = 1 - \sigma(z)$. Смена знака логита меняет классы местами: $\sigma(0) = \frac12$, $\sigma(2) \approx 0.88$, $\sigma(-2) \approx 0.12$.

**Деление степеней:** $\frac{e^a}{e^b} = e^{a-b}$, например $\frac{2^5}{2^3} = 2^2$.

**Минус перед скобкой:** $-(a - b) = -a + b = b - a$.

**Softmax:** $\mathrm{softmax}(\mathbf{z})_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$. Индексы в математике начинаются с 1, поэтому при $K = 2$ логиты называются $z_1, z_2$.

**Логит, шансы, вероятность:**
- **Логит** $z$ — сырой выход последнего слоя, любое число от $-\infty$ до $+\infty$. Его смысл — логарифм шансов: $z = \ln\frac{p}{1-p}$, а **не** $\ln p$. При $p = 0.5$: $\ln p \approx -0.69$, а логит $= \ln 1 = 0$.
- **Шансы** $\frac{p}{1-p}$ — во сколько раз класс 1 вероятнее класса 0. Пример: $p = 0.8$ → шансы $4$ → логит $\ln 4 \approx 1.39$.
- **Сигмоида и logit обратны друг другу:** $\sigma$ переводит логит в $p$, $\mathrm{logit}$ переводит $p$ обратно в логит; $\mathrm{logit}(\sigma(z)) = z$ (задача Б3.2).
- **Как читать логит:** $z = 0$ → $p = 0.5$; $z > 0$ → перевес класса 1; $z < 0$ → перевес класса 0; чем больше $|z|$, тем увереннее модель.

**Сигмоида, softmax, multi-label:**
- **Сигмоида** — выбор из двух: 1 логит → $p$, второй класс получает $1 - p$. Пример: спам или не спам.
- **Softmax** — $K$ логитов → $K$ вероятностей взаимоисключающих классов, в сумме 1. Пример: кошка, собака или птица. При $K = 2$ совпадает с $\sigma(z_1 - z_2)$.
- **Multi-label** — у объекта может быть несколько меток сразу. Ставят отдельную сигмоиду на каждый класс; вероятности независимы, сумма не обязана быть 1.
- В коде лосс считают прямо по логитам (`BCEWithLogitsLoss`, `CrossEntropyLoss`): так численно устойчивее (модуль 13).

**Путь данных в сети:**

$$\mathbf{x}\ (\text{признаки}) \xrightarrow{\text{слои } f(W\mathbf{x}+\mathbf{b})} \mathbf{h} \xrightarrow{\text{последний слой}} \mathbf{z}\ (\text{логиты}) \xrightarrow{\sigma\ /\ \text{softmax}} \mathbf{p} \xrightarrow{\text{лосс с } y} L$$

- Вход $\mathbf{x}$ — вектор чисел: пиксели, токены или столбцы таблицы.
- $W$, $\mathbf{b}$ — веса (параметры $\theta$), $f$ — нелинейность (например, ReLU).
- При обучении градиент $L$ по весам считается цепным правилом через все слои (backprop, модуль 04), затем веса сдвигаются так, чтобы $L$ уменьшился (модуль 05).

**Хвосты:** нет. Повторение закрыто: $\sigma'(0) = 0.5 \cdot 0.5 = 0.25$ — максимум производной, отсюда затухание градиента ($0.25^{10} \approx 10^{-6}$).